# Stage 9G — Document the release decision

**Goal:** Review the one final test report and record whether this model is ready for operational use.

**Why:** A high overall score can hide a serious failure for one kind of request. Release decisions must state the evidence, the risk, and what happens next.

We will read saved reports only. We will not open the test rows or change the model or its 0.20 threshold. This decision uses the observed test results; we do not pretend its review rule was set before the test.

In [ ]:
# Cell 2 — Why: verify the exact reports we are reviewing.
from google.colab import drive
from pathlib import Path
import hashlib, json

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive/nyc311_mlops")
final_id = "20260926T143549799163Z"
experiment_id = "20260926T143147282446Z"
split_id = "20260926T142339294958Z"
final_dir = root / "final_evaluations" / final_id
experiment_dir = root / "experiments" / experiment_id

report_file = final_dir / "test_report.json"
plan_file = final_dir / "locked_plan.json"
experiment_file = experiment_dir / "experiment.json"
report = json.loads(report_file.read_text())
plan = json.loads(plan_file.read_text())
experiment = json.loads(experiment_file.read_text())

if (report["final_id"] != final_id or report["split_id"] != split_id
        or report["experiment_id"] != experiment_id):
    raise ValueError("Wrong final report")
if (plan["final_id"] != final_id or plan["threshold"] != 0.2
        or plan["chosen_model"] != "tree"):
    raise ValueError("Wrong locked plan")
if experiment["experiment_id"] != experiment_id:
    raise ValueError("Wrong experiment")

print("Reviewed final test:", final_id)
print("Overall:", report["overall"])
print("Test rows were not reopened")

In [ ]:
# Cell 3 — Why: translate scores into requests people would see.
counts = report["confusion"]
heat = report["groups"]["HEAT/HOT WATER"]
snow = report["groups"]["Snow or Ice"]
brooklyn = report["groups"]["BROOKLYN"]

if sum(counts.values()) != report["test_rows"]:
    raise ValueError("Confusion counts do not add up")
if heat["late_cases"] != 552 or heat["predicted_late"] != 0:
    raise ValueError("HEAT/HOT WATER result differs from the reviewed report")

print("Late cases found:", counts["true_late"])
print("Late cases missed:", counts["missed_late"])
print("False alarms:", counts["false_alarm"])
print("HEAT/HOT WATER late cases missed:", heat["late_cases"])
print("Snow or Ice late cases:", snow["late_cases"], "— too few for a stable conclusion")
print("Brooklyn recall:", brooklyn["recall"])
print("Validation HEAT/HOT WATER recall:",
      experiment["chosen_validation_groups"]["HEAT/HOT WATER"]["recall"])
print("Test HEAT/HOT WATER recall:", heat["recall"])

In [ ]:
# Cell 4 — Why: save a traceable HOLD decision and a next experiment.
from datetime import datetime, timezone

decision_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
decision_dir = root / "release_decisions" / decision_id
decision_dir.mkdir(parents=True, exist_ok=False)

decision = {
    "decision_id": decision_id,
    "decision": "HOLD_FOR_OPERATIONAL_USE",
    "final_evaluation_id": final_id,
    "experiment_id": experiment_id,
    "split_id": split_id,
    "chosen_model": plan["chosen_model"],
    "threshold": plan["threshold"],
    "test_report_sha256": hashlib.sha256(report_file.read_bytes()).hexdigest(),
    "locked_plan_sha256": hashlib.sha256(plan_file.read_bytes()).hexdigest(),
    "experiment_sha256": hashlib.sha256(experiment_file.read_bytes()).hexdigest(),
    "evidence": {
        "overall_test": report["overall"],
        "confusion": counts,
        "heat_hot_water": heat,
        "snow_or_ice": snow,
        "brooklyn": brooklyn,
    },
    "reasons": [
        "All 552 late HEAT/HOT WATER requests in the test period were missed.",
        "Snow or Ice had only 2 late test cases, so its test recall is uncertain.",
        "Overall recall does not resolve the complaint-type failure.",
    ],
    "next_experiment": [
        "Use train and validation data to diagnose HEAT/HOT WATER by month and probability.",
        "Develop a revised model or review workflow without tuning on March–April test labels.",
        "Collect a later untouched time period for the next final evaluation.",
    ],
    "test_usage": "March–April 2026 was used for this final evaluation and is now retired for model selection.",
    "lesson_version": "stage9g-v1",
}
decision_file = decision_dir / "decision.json"
decision_file.write_text(json.dumps(decision, indent=2) + "\n", encoding="utf-8")
if json.loads(decision_file.read_text()) != decision:
    raise ValueError("Saved decision differs")

print("Decision ID:", decision_id)
print("Release decision:", decision["decision"])
print("Saved:", decision_file)
print("Next experiment:", *decision["next_experiment"], sep="\n- ")

# Stage 9G takeaway and quiz

**What you learned:** Release is a separate decision from training. We documented the model, threshold, exact evidence, reasons for the hold, and the plan for another experiment. The March–April test period has served its purpose and must not become a tuning set.

**Easy quiz:**

1. How can overall recall be high while all late HEAT/HOT WATER cases are missed?
2. Why should we be cautious about the Snow or Ice test recall of 0.50?
3. Why keep March–April out of the next model-selection process?
4. What new time period would we need to judge a revised model fairly?

Send me the **Decision ID** and **Release decision**. Next, we will diagnose the HEAT/HOT WATER failure using earlier data and design a new, fair experiment.